# Visualizing File Changes Across Commits

This notebook analyzes differences in files across multiple tardis-regression-data commits. There are two main approaches to get tardis-regression-data commits:

### Method 1: Run pytest on tardis commits and generate regression commits (False commits)

To fetch tardis commits, you have three options:

- Run pytest on latest n tardis commits
- Run pytest on str or list of multiple tardis commits

### Method 2: Directly use tardis-regression-data repo commits

To get those commits, you have two options:

- Manually provide a list of multiple tardis-regression-data commits
- Get last n tardis-regression-data commits

### Note:
By default this notebook runs pytest on latest n tardis commits and generates falsey regression commits to analyze difference.

In [1]:
from tardisbase.testing.regression_comparison.run_tests import run_tests
from tardisbase.testing.regression_comparison.visualize_files import MultiCommitCompare
from tardisbase.testing.regression_comparison.util import get_last_n_commits
import pandas as pd

INFO:numexpr.utils:Note: detected 192 virtual cores but NumExpr set to maximum of 64, check "NUMEXPR_MAX_THREADS" environment variable.
INFO:numexpr.utils:Note: NumExpr detected 192 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO:numexpr.utils:NumExpr defaulting to 16 threads.


Display Configuration

In [2]:
# Configure pandas display options for better visualization
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

Setup Configuration

In [3]:
# Configuration for the analysis
config = {
    "tardis_repo_path": "/home/riddhigangbhoj/tardis-work/tardis",
    "regression_data_repo_path": "/home/riddhigangbhoj/tardis-work/tardis-regression-data",
    "branch": "master",
    # "n": 3, # Last n commits   
    "commits": ["8dc6317d4f2df0d85b33ecebdfe75fde92cf63b8", "511f93cf9739b8e6e3406a97366bae0025438462", "20e253f9a5566ede7eeac4b1322612a547861236","aa91e497724945ca24372ae2e47b3bfbf9be1284"],  # Uncomment for specific commits
    "conda_manager": "conda"
}

## Method 1: Run pytest on tardis commits to generate falsey regression data commits

### Case 1: Test latest N TARDIS commits
Important Note: 
1. Comment out `commits` from config
2. Provide the value of `n` in config
3. To forcely recreate new enviornment each time even when enviornment already exist, do `force_recreate` as `True`
4. Either provide entire "tardis" module or selective path like "tardis/spectrum/tests/test_spectrum_solver.py" in `test_path`
5. Provide path to default current enviornment in `default_curr_env`
6. If you want to use default current enviornment only without creating new enviornment each time, do `use_new_envs` as `False`.

In [4]:
# processed_commits, regression_commits, original_head = run_tests(
#         **config,
#         force_recreate=True,
#         test_path="tardis",
#         default_curr_env="/home/riddhigangbhoj/miniforge3/envs/tardis-master",
#         use_new_envs=True
#     )

### Case 2: Test specific TARDIS commits

Important Note:
1. Comment out `n` from config
2. `commits_input` is list of `commits` hashes from config 
3. If tardis commits provided are [1,2,3,4] then the comparison would be ["2-1","3-2","4-3"] of the respective regression commits. So, make the list accordingly.
4. To forcely recreate new enviornment each time even when enviornment already exist, do `force_recreate` as `True`
5. Either provide entire "tardis" module or selective path like "tardis/spectrum/tests/test_spectrum_solver.py" in `test_path`
6. Provide path to default current enviornment in `default_curr_env`
7. If you want to use default current enviornment only without creating new enviornment each time, do `use_new_envs` as `False`.

In [4]:
processed_commits, regression_commits, original_head = run_tests(
    tardis_repo_path= "/home/riddhigangbhoj/tardis-work/tardis",
    regression_data_repo_path= "/home/riddhigangbhoj/tardis-work/tardis-regression-data",
    branch= "master", 
    commits_input=config["commits"],
    conda_manager=config["conda_manager"],
    force_recreate=True,
    test_path="tardis/spectrum/tests/test_spectrum_solver.py",
    default_curr_env="/home/riddhigangbhoj/miniforge3/envs/tardis-master",
    use_new_envs=True
)

INFO:tardisbase.testing.regression_comparison.run_tests:Original HEAD of regression data repo: 0a4e61da79baf73e5863204b014001a22aae885f
INFO:tardisbase.testing.regression_comparison.run_tests:Processing commit 1/4: 8dc6317d4f2df0d85b33ecebdfe75fde92cf63b8
INFO:tardisbase.testing.regression_comparison.run_tests:Creating conda environment: tardis-test-8dc6317d
INFO:tardisbase.testing.regression_comparison.run_tests:Checking if environment tardis-test-8dc6317d exists...
INFO:tardisbase.testing.regression_comparison.run_tests:Executing command: conda env list
INFO:tardisbase.testing.regression_comparison.run_tests:Command completed successfully. Last 3 lines of output:
INFO:tardisbase.testing.regression_comparison.run_tests:                       * /home/riddhigangbhoj/miniforge3/envs/tardis-master
INFO:tardisbase.testing.regression_comparison.run_tests:                         /home/riddhigangbhoj/miniforge3/envs/tardis-master3
INFO:tardisbase.testing.regression_comparison.run_tests:  bas

## Method 2: Use existing regression data commits

### A.  Manual Commit Selection
Note:
1. No need to run pytest for this.
2. If commits provided are [1,2,3,4] then the comparison would be ["2-1","3-2","4-3"]. So, make the list accordingly.

In [ ]:
# regression_commits = ["66a96a847c873544babb7bf934040c86433a5962",
#                       "d12d869bd2bb2038c9090852ee9ef998959f412d",
#                       "b008a7180440a697ad5b54a9f77b692d4f71b120",
#                       "a2a946a43d710c44bb3b08bcae69359fe13ed032",
#                       "9404dc594563d9457e3ba91fcaa8400cae231801"]

### B.  Automatically fetch the most recent N commits from regression data repository
Note:
1. No need to run pytest for this.
2. Set `n` to the number of recent regression commits you want to fetch.

In [ ]:
# regression_commits = get_last_n_commits(n=2, repo_path=config["regression_data_repo_path"])
# regression_commits

## Visualize File Changes
Create a visualizer object to analyze file changes across commits.
Note:
1. Uncomment and set `file_extensions` to any type of file type to filter specific files.
2. Choose `compare_function` of your choice either "git_diff" or "cmd_diff"
    - 'git_diff': Uses git's built-in diff functionality to compare files
          directly within the repository.
    - 'cmd_diff': Extracts files to temporary locations and uses the
          system's diff command.

#### Case 1: Direct regression data commits (no TARDIS commits)
Use when you are directly providing regression commits.


In [ ]:
# visualizer = MultiCommitCompare(
#     regression_repo_path=config["regression_data_repo_path"],
#     commits=regression_commits,
#     # file_extensions=('.h5', '.hdf5') # Uncomment to filter specific files
#     compare_function="git_diff"
# )


#### Case 2: Regression data commits generated from TARDIS commits
Use when you are providing tardis comits.
Note:
1. These regression commits are falsey commits(created just for testing).

In [5]:
visualizer = MultiCommitCompare(
    regression_repo_path=config["regression_data_repo_path"],
    commits=regression_commits,
    tardis_commits=processed_commits,
    tardis_repo_path=config["tardis_repo_path"],
    # file_extensions=('.h5', '.hdf5') # Uncomment to filter specific files
    compare_function="git_diff"
)

### Analyze the commits

In [6]:
visualizer.analyze_commits()

Analyzing 4 commits (3 transitions)...
Processing transition 1/3: 3c7bc2-87281e
Processing transition 2/3: 7a8bf1-3c7bc2
Processing transition 3/3: 5b41bf-7a8bf1
Found 391 total files across all transitions.


### Display the file change matrix 

In [7]:
commit_info, legend, matrix = visualizer.get_analysis_results()

### This displays the description of commits which are used to analyze differences.
Note:
1. The false commits generated from tardis commits uses description of tardis commits with prefix "Regression data for --"
2. Directly provided regression commit uses its description as it is.

In [8]:
commit_info

,Commit #,Regression Hash (first 6 chars),Description (first 60 chars),Date
0,1,87281e,Regression data for --Fix missing hyperlink for the tutorial notebooks in doc (#31,2025-08-18 03:24
1,2,3c7bc2,Regression data for --Return formal integral to `v_inner_solver_workflow.ipynb` (#,2025-08-18 03:47
2,3,7a8bf1,Regression data for --Refactor `make_source_function` into solver (#3177),2025-08-18 04:27
3,4,5b41bf,Regression data for --diataxis (#3219),2025-08-18 04:45


### Display legend

In [9]:
legend

A          Added
D        Deleted
M       Modified
•      Unchanged
−    Not-Present
Name: Legend, dtype: object

### The matrix below shows file changes across commit transitions. 
Each row represents a file, and each column represents a commit transition (e.g., "2cbdea-6483ea" means changes from commit 2cbdea to 6483ea).


In [10]:
matrix

,Files,3c7bc2-87281e,7a8bf1-3c7bc2,5b41bf-7a8bf1
0,.gitattributes,•,•,•
1,.github/actions/setup_env/action.yml,•,•,•
2,.github/workflows/run-notebook.yml,•,•,•
3,.github/workflows/trigger-lfs-cache.yml,•,•,•
4,.gitignore,•,•,•
5,LICENSE,•,•,•
6,__init__.py,•,•,•
7,arepo_data/arepo_snapshot.hdf5,•,•,•
8,arepo_data/arepo_snapshot.json,•,•,•
9,atom_data/chianti_He.h5,•,•,•


### Cleanup: Reset False Commits
**Warning**: This resets history, proceed with caution.

If you want to clean up false commits generated after running pytest, use `git reset --hard upstream/master`.Here N is the number of false commits generated.
